In [13]:
import numpy as np
import os
from collections import OrderedDict
import pandas as pd
import pickle
import time
import subprocess

from batchgenerators.dataloading.data_loader import SlimDataLoaderBase
from batchgenerators.transforms.spatial_transforms import MirrorTransform as Mirror
from batchgenerators.transforms.abstract_transforms import Compose
from batchgenerators.dataloading.multi_threaded_augmenter import MultiThreadedAugmenter
from batchgenerators.dataloading import SingleThreadedAugmenter
from batchgenerators.transforms.spatial_transforms import SpatialTransform
from batchgenerators.transforms.crop_and_pad_transforms import CenterCropTransform
from batchgenerators.transforms.utility_transforms import ConvertSegToBoundingBoxCoordinates
from experiments.exp_new_data_loader.custom_transform import RandomChannelDeleteTransform

import utils.dataloader_utils as dutils
import utils.exp_utils as utils

import argparse
import os, warnings
import time

import torch

import utils.exp_utils as utils
from evaluator import Evaluator
from predictor import Predictor
from plotting import plot_batch_prediction
import numpy as np

In [14]:
args_path = "/home/robakp/Exeriments1/prostate_lesion_detection/MDT_ProstateX/args-exp3.pkl"
with open(args_path, "rb") as f:
    args = pickle.load(f)


args.exp_dir='/home/robakp/Exeriments1/prostate_lesion_detection/MDT_ProstateX/experiments/exp_new_data_loader'
args.exp_source='/home/robakp/Exeriments1/prostate_lesion_detection/MDT_ProstateX/experiments/exp_new_data_loader'

cf = utils.prep_exp(args.exp_source, args.exp_dir, args.server_env, args.use_stored_settings)


cp: '/home/robakp/Exeriments1/prostate_lesion_detection/MDT_ProstateX/experiments/exp_new_data_loader/configs.py' and '/home/robakp/Exeriments1/prostate_lesion_detection/MDT_ProstateX/experiments/exp_new_data_loader/configs.py' are the same file


In [15]:
args

Namespace(cuda_device=0, data_dest=None, dev=False, exp_dir='/home/robakp/Exeriments1/prostate_lesion_detection/MDT_ProstateX/experiments/exp_new_data_loader', exp_source='/home/robakp/Exeriments1/prostate_lesion_detection/MDT_ProstateX/experiments/exp_new_data_loader', folds=None, last=True, mode='train', no_benchmark=False, number_of_epochs=2, resume=False, server_env=False, use_stored_settings=False, verbose=False)

In [16]:

cf.model_path = ""

In [17]:
def train(logger):
    """
    perform the training routine for a given fold. saves plots and selected parameters to the experiment dir
    specified in the configs.
    """
    logger.info('performing training in {}D over fold {} on experiment {} with model {}'.format(
        cf.dim, cf.fold, cf.exp_dir, cf.model))

    
    net = model.net(cf, logger).cuda()
    # print()
    # print()
    # print(":net:")
    # print(net)
    if hasattr(cf, "optimizer") and cf.optimizer.lower() == "adam":
        logger.info("Using Adam optimizer.")
        optimizer = torch.optim.Adam(utils.parse_params_for_optim(net, weight_decay=cf.weight_decay,
                                                                   exclude_from_wd=cf.exclude_from_wd),
                                      lr=cf.learning_rate[0])
    else:
        logger.info("Using AdamW optimizer.")
        optimizer = torch.optim.AdamW(utils.parse_params_for_optim(net, weight_decay=cf.weight_decay,
                                                                   exclude_from_wd=cf.exclude_from_wd),
                                      lr=cf.learning_rate[0])


    if cf.dynamic_lr_scheduling:
        scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode=cf.scheduling_mode, factor=cf.lr_decay_factor,
                                                               patience=cf.scheduling_patience)

    model_selector = utils.ModelSelector(cf, logger)
    train_evaluator = Evaluator(cf, logger, mode='train')
    val_evaluator = Evaluator(cf, logger, mode=cf.val_mode)

    starting_epoch = 1

    # prepare monitoring
    monitor_metrics = utils.prepare_monitoring(cf)

    if cf.resume:
        checkpoint_path = os.path.join(cf.fold_dir, "last_checkpoint")
        starting_epoch, net, optimizer, monitor_metrics = \
            utils.load_checkpoint(checkpoint_path, net, optimizer)
        logger.info('resumed from checkpoint {} to epoch {}'.format(checkpoint_path, starting_epoch))


    logger.info('loading dataset and initializing batch generators...')
    batch_gen = data_loader.get_train_generators(cf, logger)

    #It actually builds it, and then fails, but gger, mthe graph remains!
    #logger.add_graph(net.Fpn,torch.from_numpy(next(batch_gen['train'])['data']).float().cuda() )



    for epoch in range(starting_epoch, cf.num_epochs + 1):

        if epoch == starting_epoch + cf.number_of_epochs:
            break

        logger.info('starting training epoch {}'.format(epoch))
        print(f"After Epoch {epoch}:")
        print(f"Current allocated memory: {torch.cuda.memory_allocated() / 1024**2:.2f} MB")

        start_time = time.time()

        net.train()
        train_results_list = []
        #xdd
        
        for bix in range(cf.num_train_batches):
            batch = next(batch_gen['train'])
            print(batch['data'][0].shape)
            # print(f"batch {bix}")
            # print(f"Current allocated memory: 1 {torch.cuda.memory_allocated() / 1024**2:.2f} MB")
            # batch = next(batch_gen['train'])
            tic_fw = time.time()
            # print(f"Current allocated memory: 2 {torch.cuda.memory_allocated() / 1024**2:.2f} MB")
            torch.cuda.empty_cache()
            # print(f"Current allocated memory: 2.5 {torch.cuda.memory_allocated() / 1024**2:.2f} MB")
            optimizer.zero_grad()
            results_dict = net.train_forward(batch)
            # print(f"Current allocated memory: 3 {torch.cuda.memory_allocated() / 1024**2:.2f} MB")
            tic_bw = time.time()
            # print(f"Current allocated memory: 4 {torch.cuda.memory_allocated() / 1024**2:.2f} MB")
            optimizer.zero_grad()
            # print(f"Current allocated memory: 5 {torch.cuda.memory_allocated() / 1024**2:.2f} MB")
            results_dict['torch_loss'].backward()

            # print(f"Current allocated memory: 6 {torch.cuda.memory_allocated() / 1024**2:.2f} MB")
            optimizer.step()
            # print(f"Current allocated memory: 7 {torch.cuda.memory_allocated() / 1024**2:.2f} MB")
            torch.cuda.empty_cache()
            print('\rtr. batch {0}/{1} (ep. {2}) fw {3:.2f}s / bw {4:.2f} s / total {5:.2f} s || '.format(
                bix + 1, cf.num_train_batches, epoch, tic_bw - tic_fw, time.time() - tic_bw,
                time.time() - tic_fw) + results_dict['logger_string'], flush=True, end="")
            # train_results_list.append(({k:v for k,v in results_dict.items() if k != "seg_preds"}, batch["pid"]))
        print()

        _, monitor_metrics['train'] = train_evaluator.evaluate_predictions(train_results_list, monitor_metrics['train'])

        logger.info('generating training example plot.')
        utils.split_off_process(plot_batch_prediction, batch, results_dict, cf, outfile=os.path.join(
           cf.plot_dir, 'pred_example_{}_train.png'.format(cf.fold)), logger=logger)

        train_time = time.time() - start_time

        logger.info('starting validation in mode {}.'.format(cf.val_mode))
        with torch.no_grad():
            net.eval()
            print("explodea after eval?")
            iiii=0
            if cf.do_validation:
                val_results_list = []
                val_predictor = Predictor(cf, net, logger, mode='val')
                for _ in range(batch_gen['n_val']):
                    print(1)
                    iiii+=1
                    batch = next(batch_gen[cf.val_mode])
                    if cf.val_mode == 'val_patient':
                        print(2)
                        iiii+=1
                        results_dict = val_predictor.predict_patient(batch)
                        print(3)
                        iiii+=1 
                    elif cf.val_mode == 'val_sampling':
                        print(4)
                        iiii+=1
                        results_dict = net.train_forward(batch, is_validation=True)
                    print(12)
                    # val_results_list.append([results_dict['boxes'], batch['pid']])
                    val_results_list.append(({k:v for k,v in results_dict.items() if k != "seg_preds"}, batch["pid"]))
                    print(5)


                    # print(val_results_list)

                _, monitor_metrics['val'] = val_evaluator.evaluate_predictions(val_results_list, monitor_metrics['val'])
                print(6)
                model_selector.run_model_selection(net, optimizer, monitor_metrics, epoch)
                print(7)
            # update monitoring and prediction plots
            monitor_metrics.update({"lr":
                                        {str(g): group['lr'] for (g, group) in enumerate(optimizer.param_groups)}})
            logger.metrics2tboard(monitor_metrics, global_step=epoch)

            epoch_time = time.time() - start_time
            logger.info('trained epoch {}: took {} ({} train / {} val)'.format(
                epoch, utils.get_formatted_duration(epoch_time, "ms"), utils.get_formatted_duration(train_time, "ms"),
                utils.get_formatted_duration(epoch_time-train_time, "ms")))
            batch = next(batch_gen['val_sampling'])
            print(8)
            results_dict = net.train_forward(batch, is_validation=True)
            logger.info('generating validation-sampling example plot.')
            utils.split_off_process(plot_batch_prediction, batch, results_dict, cf, outfile=os.path.join(
                cf.plot_dir, 'pred_example_{}_val.png'.format(cf.fold)), logger=logger)

        if False:
            checkpoint_path = os.path.join(cf.fold_dir, f"model_epoch_{epoch}.pkl")
            logger.info(f"Saving model checkpoint at: {checkpoint_path}")
                
            torch.save({
                'epoch': epoch,
                'model_state_dict': net.state_dict(),
                'optimizer_state_dict': optimizer.state_dict(),
                'monitor_metrics': monitor_metrics,
            }, checkpoint_path)

            # ---- Clear GPU cache before reload ----
            logger.info(f"Memory pre clear {torch.cuda.memory_allocated() / 1024**2:.2f} MB")

            import gc
            del net
            del optimizer
            del monitor_metrics  # delete model instance
            gc.collect()

            torch.cuda.empty_cache()
            torch.cuda.synchronize()
            logger.info(f"Memory post clear {torch.cuda.memory_allocated() / 1024**2:.2f} MB")

            # --- Reload model from checkpoint ---
            logger.info(f"Reloading model checkpoint from: {checkpoint_path}")

            # 1️⃣ Recreate the model instance
            net = model.net(cf, logger).cuda()
            # net = MyModel(cf)  # <--- you must reinstantiate it manually

            # 2️⃣ Move it to GPU if available
            device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
            # net.to(device)

            # 3️⃣ Load checkpoint
            checkpoint = torch.load(checkpoint_path, map_location=device)
            net.load_state_dict(checkpoint['model_state_dict'])
            optimizer = torch.optim.AdamW(utils.parse_params_for_optim(net, weight_decay=cf.weight_decay,
                                    exclude_from_wd=cf.exclude_from_wd),
                                    lr=cf.learning_rate[0])
            optimizer.load_state_dict(checkpoint['optimizer_state_dict'])

            #very sus
            monitor_metrics = utils.prepare_monitoring(cf)
            # 4️⃣ Return to training mode
            net.train()
            logger.info(f"Memory post reload {torch.cuda.memory_allocated() / 1024**2:.2f} MB")
            logger.info(f"Checkpoint for epoch {epoch} successfully reloaded.")

        # -------------- scheduling -----------------
        if cf.dynamic_lr_scheduling:
            scheduler.step(monitor_metrics["val"][cf.scheduling_criterion][-1])
        else:
            for param_group in optimizer.param_groups:
                param_group['lr'] = cf.learning_rate[epoch-1]
    print(f"Current allocated memory: 1 {torch.cuda.memory_allocated() / 1024**2:.2f} MB")


In [18]:
def load_dataset(cf, logger, subset_pids=None, pp_data_path=None, pp_name=None):
    """
    loads the dataset. if deployed in cloud also copies and unpacks the data to the working directory.
    :param subset_pids: subset pids to be loaded from the dataset. used e.g. for testing to only load the test folds.
    :return: data: dictionary with one entry per patient (in this case per patient-breast, since they are treated as
    individual images for training) each entry is a dictionary containing respective meta-info as well as paths to the preprocessed
    numpy arrays to be loaded during batch-generation
    """
    if pp_data_path is None:
        pp_data_path = cf.pp_data_path
    if pp_name is None:
        pp_name = cf.pp_name
    if cf.server_env:
        copy_data = True
        target_dir = os.path.join(cf.data_dest, pp_name)
        if not os.path.exists(target_dir):
            cf.data_source_dir = pp_data_path
            os.makedirs(target_dir)
            subprocess.call('rsync -av {} {}'.format(
                os.path.join(cf.data_source_dir, cf.input_df_name), os.path.join(target_dir, cf.input_df_name)), shell=True)
            logger.info('created target dir and info df at {}'.format(os.path.join(target_dir, cf.input_df_name)))

        elif subset_pids is None:
            copy_data = False

        pp_data_path = target_dir


    p_df = pd.read_pickle(os.path.join(pp_data_path, cf.input_df_name))

    if cf.select_prototype_subset is not None:
        prototype_pids = p_df.pid.tolist()[:cf.select_prototype_subset]
        p_df = p_df[p_df.pid.isin(prototype_pids)]
        logger.warning('WARNING: using prototyping data subset!!!')

    if subset_pids is not None:
        p_df = p_df[p_df.pid.isin(subset_pids)]
        logger.info('subset: selected {} instances from df'.format(len(p_df)))

    if cf.server_env:
        if copy_data:
            copy_and_unpack_data(logger, p_df.pid.tolist(), cf.fold_dir, cf.data_source_dir, target_dir)

    class_targets = p_df['class_target'].tolist()
    pids = p_df.pid.tolist()
    imgs = [os.path.join(pp_data_path, '{}_img.npy'.format(pid)) for pid in pids]
    segs = [os.path.join(pp_data_path,'{}_rois.npy'.format(pid)) for pid in pids]

    data = OrderedDict()
    # for the experiment conducted here, malignancy scores are binarized following self.cf.modify_class_target_fn
    for ix, pid in enumerate(pids):
        targets = cf.modify_class_target_fn(class_targets[ix])
        data[pid] = {'data': imgs[ix], 'seg': segs[ix], 'pid': pid, 'class_target': targets}
        data[pid]['fg_slices'] = p_df['fg_slices'].tolist()[ix]
        #data[pid]['class_unknown']= 

    return data


In [19]:
os.path.join(args.exp_source, 'data_loader.py')

'/home/robakp/Exeriments1/prostate_lesion_detection/MDT_ProstateX/experiments/exp_new_data_loader/data_loader.py'

In [20]:
# sadsadas

In [21]:
if args.mode == 'train' or args.mode == 'train_test':
    folds = None
    cf = utils.prep_exp(args.exp_source, args.exp_dir, args.server_env, args.use_stored_settings)
    cf.debugging = True
    cf.verbose = args.verbose
    cf.last = args.last
    cf.number_of_epochs = args.number_of_epochs
    if args.dev:
        folds = [0,1]
        cf.batch_size, cf.num_epochs, cf.min_save_thresh, cf.save_n_models = 3 if cf.dim==2 else 1, 1, 0, 2
        cf.num_train_batches, cf.num_val_batches, cf.max_val_patients = 5, 1, 1
        cf.test_n_epochs =  cf.save_n_models
        cf.max_test_patients = 2
    cf.data_dest = args.data_dest
    logger = utils.get_logger(cf.exp_dir, cf.server_env)
    logger.info("cudnn benchmark: {}, deterministic: {}.".format(torch.backends.cudnn.benchmark,
                                                                 torch.backends.cudnn.deterministic))
    logger.info("sending tensors to CUDA device: {}.".format(torch.cuda.get_device_name(args.cuda_device)))
    data_loader = utils.import_module('dl', os.path.join(args.exp_source, 'data_loader.py'))
    model = utils.import_module('model', cf.model_path)
    logger.info("loaded model from {}".format(cf.model_path))
    if folds is None:
        folds = range(cf.n_cv_splits)
    with torch.cuda.device(args.cuda_device):
        for fold in folds:
            cf.fold_dir = os.path.join(cf.exp_dir, 'fold_{}'.format(fold))
            cf.fold = fold
            cf.resume = args.resume
            cf.resume = False
            if not os.path.exists(cf.fold_dir):
                os.mkdir(cf.fold_dir)
            logger.set_logfile(fold=fold)
            train(logger)
            cf.resume = False
            print("a") 
            print("a") 
            print("a") 
            print("a") 
            print("a") 
            print("a") 
            if args.mode == 'train_test' and cf.last == True:
                print("TEST TEST TEST")
                test(logger)

cp: '/home/robakp/Exeriments1/prostate_lesion_detection/MDT_ProstateX/experiments/exp_new_data_loader/configs.py' and '/home/robakp/Exeriments1/prostate_lesion_detection/MDT_ProstateX/experiments/exp_new_data_loader/configs.py' are the same file


Logging to /home/robakp/Exeriments1/prostate_lesion_detection/MDT_ProstateX/experiments/exp_new_data_loader/logs/fold_all/exec.log
cudnn benchmark: False, deterministic: False.
sending tensors to CUDA device: NVIDIA GeForce RTX 4080.
loaded model from models/retina_unet.py
performing training in 3D over fold 0 on experiment /home/robakp/Exeriments1/prostate_lesion_detection/MDT_ProstateX/experiments/exp_new_data_loader with model retina_unet
feature map shapes: [[40 40 24]
 [20 20 12]
 [10 10  6]
 [ 5  5  3]]
anchor scales: {'xy': [[4.0, 4.773932767709348, 5.697608517652258], [6.8, 8.115685705105891, 9.685934480008838], [11.559999999999999, 13.796665698680014, 16.46608861601502], [19.651999999999997, 23.454331687756024, 27.99235064722554]], 'z': [[1.3333333333333333, 1.5913109225697826, 1.8992028392174192], [2.2666666666666666, 2.7052285683686303, 3.228644826669613], [3.8533333333333326, 4.598888566226671, 5.488696205338341], [6.5506666666666655, 7.818110562585341, 9.330783549075178]]}

/home/robakp/Exeriments1/prostate_lesion_detection/venv_prostate_try_2/lib/python3.7/site-packages/batchgenerators-0.20.1-py3.7.egg/batchgenerators/augmentations/utils.py:564: VisibleDeprecationWarning: Creating an ndarray from ragged nested sequences (which is a list-or-tuple of lists-or-tuples-or ndarrays with different lengths or shapes) is deprecated. If you meant to do this, you must specify 'dtype=object' when creating the ndarray.
  data_dict['bb_target'] = np.array(bb_target)


(4, 160, 160, 24)
torch.Size([6, 4, 160, 160, 24])
tr. batch 1/1 (ep. 1) fw 2.40s / bw 0.73 s / total 3.13 s || loss: 3.92, class: 1.50, bbox: 1.30, seg dice: 0.997, seg ce: 1.257, mean pix. pr.: 0.79855
evaluating in mode train
evaluating with match_iou: 1e-05
start metrics calculations

dict_keys([1, 2, 3, 4])
cl: 1
score level patient
Empty DataFrame
Columns: [pred_score, class_label, pred_class, pid, det_type, fold, match_iou]
Index: []
clu Series([], Name: class_label, dtype: float64)
Empty DataFrame
Columns: [pid, class_label, pred_score, fold]
Index: []
{'name': 'fold_0 patient cl_1', 'auc': nan, 'roc': nan, 'ap': nan, 'prc': nan, 'mean_auc': nan, 'mean_ap': nan}
score level rois
{'name': 'fold_0 rois cl_1', 'ap': nan, 'auc': nan, 'roc': nan, 'prc': nan, 'mean_ap': nan, 'mean_auc': 0}
cl: 2
score level patient
Empty DataFrame
Columns: [pred_score, class_label, pred_class, pid, det_type, fold, match_iou]
Index: []
clu Series([], Name: class_label, dtype: float64)
Empty DataFrame


/home/robakp/Exeriments1/prostate_lesion_detection/venv_prostate_try_2/lib/python3.7/site-packages/numpy/core/fromnumeric.py:3441: RuntimeWarning: Mean of empty slice.
  out=out, **kwargs)
/home/robakp/Exeriments1/prostate_lesion_detection/venv_prostate_try_2/lib/python3.7/site-packages/numpy/core/_methods.py:189: RuntimeWarning: invalid value encountered in double_scalars
  ret = ret.dtype.type(ret / rcount)
/home/robakp/Exeriments1/prostate_lesion_detection/venv_prostate_try_2/lib/python3.7/site-packages/pandas/core/ops/__init__.py:1115: FutureWarning: elementwise comparison failed; returning scalar instead, but in the future will perform elementwise comparison
  result = method(y)


starting validation in mode val_sampling.
explodea after eval?
1
Number of patients: 8
Patient IDs: ['389', '3339', '1093', '495', '378', '355', '3041', '1184']
class_targets_list: [[0], [0], [0], [1], [1], [1], [0], [0]]
len(class_targets_list): 8
batch_size: 6
inside loader hihi
(4, 160, 160, 24)
(1, 160, 160, 24)
Before cropping
data.shape: (4, 160, 160, 24)
seg.shape : (1, 160, 160, 24)
crop_dims: []
inside loader hihi
(4, 160, 160, 24)
(1, 160, 160, 24)
Before cropping
data.shape: (4, 160, 160, 24)
seg.shape : (1, 160, 160, 24)
crop_dims: []
inside loader hihi
(4, 160, 160, 24)
(1, 160, 160, 24)
Before cropping
data.shape: (4, 160, 160, 24)
seg.shape : (1, 160, 160, 24)
crop_dims: []
inside loader hihi
(4, 160, 160, 24)
(1, 160, 160, 24)
Before cropping
data.shape: (4, 160, 160, 24)
seg.shape : (1, 160, 160, 24)
crop_dims: []
inside loader hihi
(4, 160, 160, 24)
(1, 160, 160, 24)
Before cropping
data.shape: (4, 160, 160, 24)
seg.shape : (1, 160, 160, 24)
crop_dims: []
inside loade

IndexError: index 1 is out of bounds for axis 0 with size 1

In [ ]:
def get_train_generators(cf, logger):
    """
    wrapper function for creating the training batch generator pipeline. returns the train/val generators.
    selects patients according to cv folds (generated by first run/fold of experiment):
    splits the data into n-folds, where 1 split is used for val, 1 split for testing and the rest for training. 
    (inner loop test set)
    """
        
    #Generate info_df.pickle when training for the first time
    files = [os.path.join(cf.pp_dir, f) for f in os.listdir(cf.pp_dir) if 'meta_info' in f]
    df = pd.DataFrame(columns=['pid', 'class_target', 'spacing', 'fg_slices'], dtype=object)
    for f in files:
        with open(f, 'rb') as handle:
            df.loc[len(df)] = pickle.load(handle)
    df.to_pickle(os.path.join(cf.pp_dir, 'info_df.pickle'))
    print ("Aggregated meta info to df with length", len(df))
    
    #load data
    all_data = load_dataset(cf, logger)
    splits_file = os.path.join(cf.exp_dir, 'fold_ids.pickle')
    ss= ss_v2
        
    #Get subset idx
    pids= list(pd.read_pickle(os.path.join(cf.pp_dir, 'info_df.pickle')).pid.values)

    #Regenerate fold_ids.pickle?
    if not os.path.isfile(splits_file) and not cf.created_fold_id_pickle:           
        #Get subsets
        prostatex_train_fix, prostatex_train_val, prostatex_test= ss['train_fix'] if cf.use_prostatex_test else [], \
                                                                  ss['train_val'], ss['test']
        #Build samples for all folds     
        from sklearn.model_selection import KFold

        prostatex_kf= KFold(n_splits=cf.n_cv_splits, random_state=5, shuffle=True)
        prostatex_kf.get_n_splits(prostatex_train_val)

        fg= []
        for i, (prostatex_train_index, prostatex_val_index) in \
            enumerate(prostatex_kf.split(prostatex_train_val)) :
            fg.append([
                list(np.array(prostatex_train_val)[prostatex_train_index]) + prostatex_train_fix,
                list(np.array(prostatex_train_val)[prostatex_val_index]),
                prostatex_test,
                i
            ])

        #Save it
        pickle.dump(fg, open(os.path.join(cf.exp_dir, 'fold_ids.pickle'), 'wb'))
        cf.created_fold_id_pickle = True
    else:
        with open(splits_file, 'rb') as handle:
            fg = pickle.load(handle)

    train_pids, val_pids, test_pids, _ = fg[cf.fold]
    
    if cf.verbose == True:
        print('Train IDs:', train_pids)
        print('\nValidation IDs:', val_pids)
        print('\nTest IDs:', test_pids)

    train_data = {k: v for (k, v) in all_data.items() if any(p == v['pid'] for p in train_pids)}
    val_data = {k: v for (k, v) in all_data.items() if any(p == v['pid'] for p in val_pids)}

    logger.info("data set loaded with: {} train / {} val / {} test patients".format(len(train_pids), len(val_pids), len(test_pids)))
    batch_gen = {}
    batch_gen['train'] = create_data_gen_pipeline(train_data, cf=cf, is_training=True)
    batch_gen['val_sampling'] = create_data_gen_pipeline(val_data, cf=cf, is_training=False)
    if cf.val_mode == 'val_patient':
        batch_gen['val_patient'] = PatientBatchIterator(val_data, cf=cf)
        batch_gen['n_val'] = len(val_pids) if cf.max_val_patients is None else min(len(val_pids), cf.max_val_patients)
    else:
        batch_gen['n_val'] = cf.num_val_batches

    return batch_gen


testing data load 1

In [ ]:
folds = None
cf = utils.prep_exp(args.exp_source, args.exp_dir, args.server_env, args.use_stored_settings)
cf.verbose = args.verbose
cf.last = args.last
cf.number_of_epochs = args.number_of_epochs
if args.dev:
    folds = [0,1]
    cf.batch_size, cf.num_epochs, cf.min_save_thresh, cf.save_n_models = 3 if cf.dim==2 else 1, 1, 0, 2
    cf.num_train_batches, cf.num_val_batches, cf.max_val_patients = 5, 1, 1
    cf.test_n_epochs =  cf.save_n_models
    cf.max_test_patients = 2
cf.data_dest = args.data_dest
logger = utils.get_logger(cf.exp_dir, cf.server_env)
logger.info("cudnn benchmark: {}, deterministic: {}.".format(torch.backends.cudnn.benchmark,
                                                             torch.backends.cudnn.deterministic))
logger.info("sending tensors to CUDA device: {}.".format(torch.cuda.get_device_name(args.cuda_device)))
data_loader = utils.import_module('dl', os.path.join(args.exp_source, 'data_loader.py'))
model = utils.import_module('model', cf.model_path)
logger.info("loaded model from {}".format(cf.model_path))
if folds is None:
    folds = range(cf.n_cv_splits)



cp: '/home/robakp/Exeriments1/prostate_lesion_detection/MDT_ProstateX/experiments/exp_new_data_loader/configs.py' and '/home/robakp/Exeriments1/prostate_lesion_detection/MDT_ProstateX/experiments/exp_new_data_loader/configs.py' are the same file


Logging to /home/robakp/Exeriments1/prostate_lesion_detection/MDT_ProstateX/experiments/exp_new_data_loader/logs/fold_all/exec.log
cudnn benchmark: False, deterministic: False.
cudnn benchmark: False, deterministic: False.
sending tensors to CUDA device: NVIDIA GeForce RTX 4080.
sending tensors to CUDA device: NVIDIA GeForce RTX 4080.
loaded model from models/retina_unet.py
loaded model from models/retina_unet.py


get train generators trials

In [ ]:
class BatchGenerator(SlimDataLoaderBase):
    """
    creates the training/validation batch generator. Samples n_batch_size patients (draws a slice from each patient if 2D)
    from the data set while maintaining foreground-class balance. Returned patches are cropped/padded to pre_crop_size.
    Actual patch_size is obtained after data augmentation.
    :param data: data dictionary as provided by 'load_dataset'.
    :param batch_size: number of patients to sample for the batch
    :return dictionary containing the batch data (b, c, y, x(, z)) / seg (b, 1, y, x(, z)) / pids / class_target
    """
    def __init__(self, data, batch_size, cf):
        super(BatchGenerator, self).__init__(data, batch_size)
        
        self.cf = cf
        self.crop_margin = np.array(self.cf.patch_size)/8. #min distance of ROI center to edge of cropped_patch.
        self.p_fg = 0.5

    def generate_train_batch(self):


        print("generating training batch")

        batch_data, batch_segs, batch_pids, batch_targets, batch_patient_labels = [], [], [], [], []
        class_targets_list =  [v['class_target'] for (k, v) in self._data.items()]

        #I am turning this off, because it is problematic with my class 20
        if False: #self.cf.head_classes > 2:
            # samples patients towards equilibrium of foreground classes on a roi-level (after randomly sampling the ratio "batch_sample_slack).
            batch_ixs = dutils.get_class_balanced_patients(
                class_targets_list, self.batch_size, self.cf.head_classes - 1, slack_factor=self.cf.batch_sample_slack)
        else:
            batch_ixs = np.random.choice(len(class_targets_list), self.batch_size)

        patients = list(self._data.items())

        for b in batch_ixs:
            patient = patients[b][1]

            # data shape: from (c, z, y, x) to (c, y, x, z).
            data = np.transpose(np.load(patient['data'], mmap_mode='r'), axes=(3, 1, 2, 0))
            seg = np.transpose(np.load(patient['seg'], mmap_mode='r'), axes=(3, 1, 2, 0))
            batch_pids.append(patient['pid'])
            batch_targets.append(patient['class_target'])

            if self.cf.dim == 2:
                # draw random slice from patient while oversampling slices containing foreground objects with p_fg.
                if len(patient['fg_slices']) > 0:
                    fg_prob = self.p_fg / len(patient['fg_slices'])
                    bg_prob = (1 - self.p_fg) / (data.shape[3] - len(patient['fg_slices']))
                    slices_prob = [fg_prob if ix in patient['fg_slices'] else bg_prob for ix in range(data.shape[3])]
                    slice_id = np.random.choice(data.shape[3], p=slices_prob)
                else:
                    slice_id = np.random.choice(data.shape[3])

                # if set to not None, add neighbouring slices to each selected slice in channel dimension.
                if self.cf.n_3D_context is not None:
                    padded_data = dutils.pad_nd_image(data[0], [(data.shape[-1] + (self.cf.n_3D_context*2))], mode='constant')
                    padded_slice_id = slice_id + self.cf.n_3D_context
                    data = (np.concatenate([padded_data[..., ii][np.newaxis] for ii in range(
                        padded_slice_id - self.cf.n_3D_context, padded_slice_id + self.cf.n_3D_context + 1)], axis=0))
                else:
                    data = data[..., slice_id]
                seg = seg[..., slice_id]

            # pad data if smaller than pre_crop_size.
            if np.any([data.shape[dim + 1] < ps for dim, ps in enumerate(self.cf.pre_crop_size)]):
                new_shape = [np.max([data.shape[dim + 1], ps]) for dim, ps in enumerate(self.cf.pre_crop_size)]
                data = dutils.pad_nd_image(data, new_shape, mode='constant')
                seg = dutils.pad_nd_image(seg, new_shape, mode='constant')

            # crop patches of size pre_crop_size, while sampling patches containing foreground with p_fg.
            crop_dims = [dim for dim, ps in enumerate(self.cf.pre_crop_size) if data.shape[dim + 1] > ps]
            if len(crop_dims) > 0:
                fg_prob_sample = np.random.rand(1)
                # with p_fg: sample random pixel from random ROI and shift center by random value.
                if fg_prob_sample < self.p_fg and np.sum(seg) > 0:
                    seg_ixs = np.argwhere(seg == np.random.choice(np.unique(seg)[1:], 1))
                    roi_anchor_pixel = seg_ixs[np.random.choice(seg_ixs.shape[0], 1)][0]
                    assert seg[tuple(roi_anchor_pixel)] > 0
                    # sample the patch center coords. constrained by edges of images - pre_crop_size /2. And by
                    # distance to the desired ROI < patch_size /2.
                    # (here final patch size to account for center_crop after data augmentation).
                    sample_seg_center = {}
                    for ii in crop_dims:
                        low = np.max((self.cf.pre_crop_size[ii]//2, roi_anchor_pixel[ii] - (self.cf.patch_size[ii]//2 - self.crop_margin[ii])))
                        high = np.min((data.shape[ii + 1] - self.cf.pre_crop_size[ii]//2,
                                       roi_anchor_pixel[ii] + (self.cf.patch_size[ii]//2 - self.crop_margin[ii])))
                        # happens if lesion on the edge of the image. dont care about roi anymore,
                        # just make sure pre-crop is inside image.
                        if low >= high:
                            low = data.shape[ii + 1] // 2 - (data.shape[ii + 1] // 2 - self.cf.pre_crop_size[ii] // 2)
                            high = data.shape[ii + 1] // 2 + (data.shape[ii + 1] // 2 - self.cf.pre_crop_size[ii] // 2)
                        sample_seg_center[ii] = np.random.randint(low=low, high=high)

                else:
                    # not guaranteed to be empty. probability of emptiness depends on the data.
                    sample_seg_center = {ii: np.random.randint(low=self.cf.pre_crop_size[ii]//2,
                                                           high=data.shape[ii + 1] - self.cf.pre_crop_size[ii]//2) for ii in crop_dims}

                for ii in crop_dims:
                    min_crop = int(sample_seg_center[ii] - self.cf.pre_crop_size[ii] // 2)
                    max_crop = int(sample_seg_center[ii] + self.cf.pre_crop_size[ii] // 2)
                    data = np.take(data, indices=range(min_crop, max_crop), axis=ii + 1)
                    seg = np.take(seg, indices=range(min_crop, max_crop), axis=ii)

            batch_data.append(data)
            batch_segs.append(seg)

        data = np.array(batch_data)
        seg = np.array(batch_segs).astype(np.uint8)
        class_target = np.array(batch_targets, dtype=object)
        return {'data': data, 'seg': seg, 'pid': batch_pids, 'class_target': class_target}


In [ ]:
ss= {
    'train_fix': ['ProstateX-0204', 'ProstateX-0205', 'ProstateX-0206', 'ProstateX-0207', 'ProstateX-0208', 'ProstateX-0209', 'ProstateX-0210', 'ProstateX-0211', 'ProstateX-0212', 'ProstateX-0213', 'ProstateX-0214', 'ProstateX-0215', 'ProstateX-0216', 'ProstateX-0217', 'ProstateX-0218', 'ProstateX-0219', 'ProstateX-0220', 'ProstateX-0221', 'ProstateX-0222', 'ProstateX-0223', 'ProstateX-0224', 'ProstateX-0225', 'ProstateX-0226', 'ProstateX-0227', 'ProstateX-0228', 'ProstateX-0229', 'ProstateX-0230', 'ProstateX-0231', 'ProstateX-0232', 'ProstateX-0233', 'ProstateX-0234', 'ProstateX-0235', 'ProstateX-0236', 'ProstateX-0237', 'ProstateX-0238', 'ProstateX-0239', 'ProstateX-0240', 'ProstateX-0241', 'ProstateX-0242', 'ProstateX-0243', 'ProstateX-0244', 'ProstateX-0245', 'ProstateX-0246', 'ProstateX-0247', 'ProstateX-0248', 'ProstateX-0249', 'ProstateX-0250', 'ProstateX-0251', 'ProstateX-0252', 'ProstateX-0253', 'ProstateX-0254', 'ProstateX-0255', 'ProstateX-0256', 'ProstateX-0257', 'ProstateX-0258', 'ProstateX-0259', 'ProstateX-0260', 'ProstateX-0261', 'ProstateX-0262', 'ProstateX-0263', 'ProstateX-0264', 'ProstateX-0265', 'ProstateX-0266', 'ProstateX-0267', 'ProstateX-0268', 'ProstateX-0269', 'ProstateX-0270', 'ProstateX-0271', 'ProstateX-0272', 'ProstateX-0273', 'ProstateX-0274', 'ProstateX-0275', 'ProstateX-0276', 'ProstateX-0277', 'ProstateX-0278', 'ProstateX-0279', 'ProstateX-0280', 'ProstateX-0281', 'ProstateX-0282', 'ProstateX-0283', 'ProstateX-0284', 'ProstateX-0285', 'ProstateX-0286', 'ProstateX-0287', 'ProstateX-0288', 'ProstateX-0289', 'ProstateX-0290', 'ProstateX-0291', 'ProstateX-0292', 'ProstateX-0293', 'ProstateX-0294', 'ProstateX-0295', 'ProstateX-0296', 'ProstateX-0297', 'ProstateX-0298', 'ProstateX-0299', 'ProstateX-0300', 'ProstateX-0301', 'ProstateX-0302', 'ProstateX-0303', 'ProstateX-0304', 'ProstateX-0305', 'ProstateX-0306', 'ProstateX-0307', 'ProstateX-0308', 'ProstateX-0309', 'ProstateX-0310', 'ProstateX-0311', 'ProstateX-0312', 'ProstateX-0313', 'ProstateX-0314', 'ProstateX-0315', 'ProstateX-0316', 'ProstateX-0317', 'ProstateX-0318', 'ProstateX-0319', 'ProstateX-0320', 'ProstateX-0321', 'ProstateX-0322', 'ProstateX-0323', 'ProstateX-0324', 'ProstateX-0326', 'ProstateX-0327', 'ProstateX-0328', 'ProstateX-0329', 'ProstateX-0330', 'ProstateX-0332', 'ProstateX-0333', 'ProstateX-0334', 'ProstateX-0335', 'ProstateX-0336', 'ProstateX-0337', 'ProstateX-0338', 'ProstateX-0339', 'ProstateX-0340', 'ProstateX-0341', 'ProstateX-0342', 'ProstateX-0343', 'ProstateX-0344', 'ProstateX-0345'], 
     'train_val': ['ProstateX-0000', 'ProstateX-0002', 'ProstateX-0004', 'ProstateX-0005', 'ProstateX-0006', 'ProstateX-0007', 'ProstateX-0008', 'ProstateX-0009', 'ProstateX-0011', 'ProstateX-0012', 'ProstateX-0013', 'ProstateX-0014', 'ProstateX-0015', 'ProstateX-0016', 'ProstateX-0017', 'ProstateX-0019', 'ProstateX-0020', 'ProstateX-0021', 'ProstateX-0023', 'ProstateX-0024', 'ProstateX-0025', 'ProstateX-0027', 'ProstateX-0028', 'ProstateX-0029', 'ProstateX-0030', 'ProstateX-0031', 'ProstateX-0033', 'ProstateX-0035', 'ProstateX-0037', 'ProstateX-0038', 'ProstateX-0040', 'ProstateX-0041', 'ProstateX-0042', 'ProstateX-0043', 'ProstateX-0044', 'ProstateX-0046', 'ProstateX-0047', 'ProstateX-0049', 'ProstateX-0050', 'ProstateX-0051', 'ProstateX-0053', 'ProstateX-0054', 'ProstateX-0056', 'ProstateX-0058', 'ProstateX-0059', 'ProstateX-0060', 'ProstateX-0063', 'ProstateX-0064', 'ProstateX-0065', 'ProstateX-0066', 'ProstateX-0067', 'ProstateX-0068', 'ProstateX-0069', 'ProstateX-0070', 'ProstateX-0071', 'ProstateX-0072', 'ProstateX-0075', 'ProstateX-0078', 'ProstateX-0080', 'ProstateX-0081', 'ProstateX-0082', 'ProstateX-0083', 'ProstateX-0084', 'ProstateX-0085', 'ProstateX-0086', 'ProstateX-0087', 'ProstateX-0088', 'ProstateX-0089', 'ProstateX-0090', 'ProstateX-0091', 'ProstateX-0092', 'ProstateX-0093', 'ProstateX-0094', 'ProstateX-0095', 'ProstateX-0096', 'ProstateX-0097', 'ProstateX-0098', 'ProstateX-0099', 'ProstateX-0100', 'ProstateX-0101', 'ProstateX-0102', 'ProstateX-0103', 'ProstateX-0104', 'ProstateX-0105', 'ProstateX-0106', 'ProstateX-0107', 'ProstateX-0108', 'ProstateX-0109', 'ProstateX-0110', 'ProstateX-0111', 'ProstateX-0112', 'ProstateX-0116', 'ProstateX-0117', 'ProstateX-0120', 'ProstateX-0121', 'ProstateX-0122', 'ProstateX-0123', 'ProstateX-0124', 'ProstateX-0125', 'ProstateX-0126', 'ProstateX-0128', 'ProstateX-0129', 'ProstateX-0131', 'ProstateX-0132', 'ProstateX-0133', 'ProstateX-0135', 'ProstateX-0136', 'ProstateX-0140', 'ProstateX-0141', 'ProstateX-0143', 'ProstateX-0144', 'ProstateX-0145', 'ProstateX-0148', 'ProstateX-0149', 'ProstateX-0150', 'ProstateX-0151', 'ProstateX-0152', 'ProstateX-0153', 'ProstateX-0155', 'ProstateX-0156', 'ProstateX-0158', 'ProstateX-0160', 'ProstateX-0162', 'ProstateX-0163', 'ProstateX-0165', 'ProstateX-0167', 'ProstateX-0168', 'ProstateX-0169', 'ProstateX-0171', 'ProstateX-0172', 'ProstateX-0173', 'ProstateX-0174', 'ProstateX-0175', 'ProstateX-0176', 'ProstateX-0177', 'ProstateX-0178', 'ProstateX-0179', 'ProstateX-0180', 'ProstateX-0181', 'ProstateX-0182', 'ProstateX-0183', 'ProstateX-0184', 'ProstateX-0185', 'ProstateX-0186', 'ProstateX-0187', 'ProstateX-0188', 'ProstateX-0189', 'ProstateX-0190', 'ProstateX-0191', 'ProstateX-0192', 'ProstateX-0193', 'ProstateX-0194', 'ProstateX-0195', 'ProstateX-0196', 'ProstateX-0197', 'ProstateX-0198', 'ProstateX-0199', 'ProstateX-0201', 'ProstateX-0203'],  
    'test': ['ProstateX-0001', 'ProstateX-0003', 'ProstateX-0010', 'ProstateX-0018', 'ProstateX-0022', 'ProstateX-0026', 'ProstateX-0032', 'ProstateX-0034', 'ProstateX-0036', 'ProstateX-0039', 'ProstateX-0045', 'ProstateX-0048', 'ProstateX-0052', 'ProstateX-0055', 'ProstateX-0057', 'ProstateX-0061', 'ProstateX-0062', 'ProstateX-0073', 'ProstateX-0074', 'ProstateX-0076', 'ProstateX-0077', 'ProstateX-0079', 'ProstateX-0113', 'ProstateX-0114', 'ProstateX-0115', 'ProstateX-0118', 'ProstateX-0119', 'ProstateX-0127', 'ProstateX-0130', 'ProstateX-0134', 'ProstateX-0137', 'ProstateX-0138', 'ProstateX-0139', 'ProstateX-0142', 'ProstateX-0146', 'ProstateX-0147', 'ProstateX-0154', 'ProstateX-0157', 'ProstateX-0159', 'ProstateX-0161', 'ProstateX-0164', 'ProstateX-0166', 'ProstateX-0170', 'ProstateX-0200', 'ProstateX-0202']
       }

In [ ]:
"""
wrapper function for creating the training batch generator pipeline. returns the train/val generators.
selects patients according to cv folds (generated by first run/fold of experiment):
splits the data into n-folds, where 1 split is used for val, 1 split for testing and the rest for training. 
(inner loop test set)
"""
    
#Generate info_df.pickle when training for the first time
files = [os.path.join(cf.pp_dir, f) for f in os.listdir(cf.pp_dir) if 'meta_info' in f]
df = pd.DataFrame(columns=['pid', 'class_target', 'spacing', 'fg_slices'], dtype=object)
for f in files:
    with open(f, 'rb') as handle:
        df.loc[len(df)] = pickle.load(handle)
df.to_pickle(os.path.join(cf.pp_dir, 'info_df.pickle'))
print ("Aggregated meta info to df with length", len(df))

#load data
all_data = load_dataset(cf, logger)
splits_file = os.path.join(cf.exp_dir, 'fold_ids.pickle')
    
#Get subset idx
pids= list(pd.read_pickle(os.path.join(cf.pp_dir, 'info_df.pickle')).pid.values)
#Regenerate fold_ids.pickle?
if not os.path.isfile(splits_file) and not cf.created_fold_id_pickle:           
    #Get subsets
    prostatex_train_fix, prostatex_train_val, prostatex_test= ss['train_fix'] if cf.use_prostatex_test else [], \
                                                              ss['train_val'], ss['test']
    #Build samples for all folds     
    from sklearn.model_selection import KFold
    prostatex_kf= KFold(n_splits=cf.n_cv_splits, random_state=5, shuffle=True)
    prostatex_kf.get_n_splits(prostatex_train_val)
    fg= []
    for i, (prostatex_train_index, prostatex_val_index) in \
        enumerate(prostatex_kf.split(prostatex_train_val)) :
        fg.append([
            list(np.array(prostatex_train_val)[prostatex_train_index]) + prostatex_train_fix,
            list(np.array(prostatex_train_val)[prostatex_val_index]),
            prostatex_test,
            i
        ])
    #Save it
    pickle.dump(fg, open(os.path.join(cf.exp_dir, 'fold_ids.pickle'), 'wb'))
    cf.created_fold_id_pickle = True
else:
    with open(splits_file, 'rb') as handle:
        fg = pickle.load(handle)
train_pids, val_pids, test_pids, _ = fg[0]

if True:
    print('Train IDs:', train_pids)
    print('\nValidation IDs:', val_pids)
    print('\nTest IDs:', test_pids)
train_data = {k: v for (k, v) in all_data.items() if any(p == v['pid'] for p in train_pids)}
val_data = {k: v for (k, v) in all_data.items() if any(p == v['pid'] for p in val_pids)}
logger.info("data set loaded with: {} train / {} val / {} test patients".format(len(train_pids), len(val_pids), len(test_pids)))
batch_gen = {}
# batch_gen['train'] = create_data_gen_pipeline(train_data, cf=cf, is_training=True)
# batch_gen['val_sampling'] = create_data_gen_pipeline(val_data, cf=cf, is_training=False)
# if cf.val_mode == 'val_patient':
#     batch_gen['val_patient'] = PatientBatchIterator(val_data, cf=cf)
#     batch_gen['n_val'] = len(val_pids) if cf.max_val_patients is None else min(len(val_pids), cf.max_val_patients)
# else:
#     batch_gen['n_val'] = cf.num_val_batches


Aggregated meta info to df with length 380


NameError: name 'ss' is not defined

In [ ]:
patient_data = train_data
cf=cf
is_training=True
"""
create mutli-threaded train/val/test batch generation and augmentation pipeline.
:param patient_data: dictionary containing one dictionary per patient in the train/test subset.
:param is_training: (optional) whether to perform data augmentation (training) or not (validation/testing)
:return: multithreaded_generator
"""

# create instance of batch generator as first element in pipeline.
data_gen = BatchGenerator(patient_data, batch_size=cf.batch_size, cf=cf)
# add transformations to pipeline.
my_transforms = []
if is_training:
    mirror_transform = Mirror(axes=cf.mirror_axes) #np.arange(cf.dim)
    my_transforms.append(mirror_transform)
    spatial_transform = SpatialTransform(patch_size=cf.patch_size[:cf.dim],
                                         patch_center_dist_from_border=cf.da_kwargs['rand_crop_dist'],
                                         do_elastic_deform=cf.da_kwargs['do_elastic_deform'],
                                         alpha=cf.da_kwargs['alpha'], sigma=cf.da_kwargs['sigma'],
                                         do_rotation=cf.da_kwargs['do_rotation'], angle_x=cf.da_kwargs['angle_x'],
                                         angle_y=cf.da_kwargs['angle_y'], angle_z=cf.da_kwargs['angle_z'],
                                         do_scale=cf.da_kwargs['do_scale'], scale=cf.da_kwargs['scale'],
                                         random_crop=cf.da_kwargs['random_crop'])
    my_transforms.append(spatial_transform)
    my_transforms.append(RandomChannelDeleteTransform(cf.droppable_channels, cf.channel_drop_p))
else:
    my_transforms.append(CenterCropTransform(crop_size=cf.patch_size[:cf.dim]))
my_transforms.append(ConvertSegToBoundingBoxCoordinates(cf.dim, get_rois_from_seg_flag=False, class_specific_seg_flag=cf.class_specific_seg_flag))
all_transforms = Compose(my_transforms)
if cf.debugging:
    multithreaded_generator = SingleThreadedAugmenter(data_gen, all_transforms)
else:
    multithreaded_generator = MultiThreadedAugmenter(data_gen, all_transforms, num_processes=cf.n_workers, seeds=range(cf.n_workers))

In [ ]:
batch = data_gen.generate_train_batch()
batch

{'data': data, 'seg': seg, 'pid': batch_pids, 'class_target': class_target} 

In [ ]:
batch['data'].shape, batch['seg'].shape, batch['pid'], batch['class_target']

In [ ]:
train_data

In [ ]:
cf.dim

In [ ]:
print("pre_crop_size:", cf.pre_crop_size)
print("patch_size:", cf.patch_size)

pre_crop_size: [160, 160, 24]
patch_size: [160, 160, 24]


<function convert_seg_to_bounding_box_coordinates at 0x76eee73b9320>
batchgenerators.augmentations.utils
/home/robakp/Exeriments1/prostate_lesion_detection/venv_prostate_try_2/lib/python3.7/site-packages/batchgenerators-0.20.1-py3.7.egg/batchgenerators/augmentations/utils.py
